# 7.3 Transforming Higher Ed Text Data to Vectors — Code Brief

## Key Concepts

- Ordinal (Likert 1-4) survey items are already numeric — usable directly. Free-response text requires vectorization.
- **CountVectorizer** — raw word-frequency (Bag-of-Words). Simple, interpretable, ignores word order.
- **TF-IDF** — weights words by distinctiveness across the corpus. Better discriminatory power than raw counts.
- Final step: merge TF-IDF vectors with the 10 NSSE ordinal columns into one analysis-ready matrix.

## Setup and Data Preparation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np


pd.options.display.max_columns = None

In [ ]:
filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'
ML_Survey_Data = pd.read_csv(f'{filepath}ML_Survey_Data.csv')
ML_Survey_Data22 = pd.read_csv(f'{filepath}ML_Survey_Data22.csv')
display(ML_Survey_Data)

## Vectorization with CountVectorizer

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Instantiate CountVectorizer with specified parameters
count_vectorizer = CountVectorizer(
    stop_words='english',
    lowercase=True,
    ngram_range=(1, 1)
)

# Apply CountVectorizer to the 'Free_Response_Text' column
count_matrix = count_vectorizer.fit_transform(ML_Survey_Data['Free_Response_Text'])

# Get feature names (vocabulary)
count_feature_names = count_vectorizer.get_feature_names_out()

# Convert the sparse matrix to a pandas DataFrame
df_count_vectorized = pd.DataFrame(count_matrix.toarray(), columns=count_feature_names)

# Display the first few rows and columns of the CountVectorized DataFrame
print("First few rows of CountVectorized DataFrame:")
df_count_vectorized.index = ML_Survey_Data.index
df_count_vectorized
print(f"\nShape of CountVectorized DataFrame: {df_count_vectorized.shape}")

## Vectorization with TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Instantiate TfidfVectorizer with specified parameters
tfidf_vectorizer = TfidfVectorizer(
    stop_words='english',
    lowercase=True,
    ngram_range=(1, 1)
)

# Apply TfidfVectorizer to the 'Free_Response_Text' column
tfidf_matrix = tfidf_vectorizer.fit_transform(ML_Survey_Data['Free_Response_Text'])

# Get feature names (vocabulary)
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

# Convert the sparse matrix to a pandas DataFrame
df_tfidf_vectorized = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_feature_names)

# Display the first few rows and columns of the TfidfVectorized DataFrame
print("First few rows of TfidfVectorized DataFrame:")
df_tfidf_vectorized.index = ML_Survey_Data.index
df_tfidf_vectorized
print(f"\nShape of TfidfVectorized DataFrame: {df_tfidf_vectorized.shape}")

## Helper Function for Text Vectorization

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

def vectorize_text_data(dataframe):
    """
    Applies CountVectorizer and TfidfVectorizer to the 'Free_Response_Text' column
    of a given DataFrame.

    Args:
        dataframe (pd.DataFrame): The input DataFrame containing a 'Free_Response_Text' column.

    Returns:
        tuple: A tuple containing two pandas DataFrames:
               - df_count_vectorized: DataFrame with CountVectorizer results.
               - df_tfidf_vectorized: DataFrame with TfidfVectorizer results.
    """
    # Instantiate CountVectorizer with specified parameters
    count_vectorizer = CountVectorizer(
        stop_words='english',
        lowercase=True,
        ngram_range=(1, 1)
    )

    # Apply CountVectorizer
    count_matrix = count_vectorizer.fit_transform(dataframe['Free_Response_Text'])
    count_feature_names = count_vectorizer.get_feature_names_out()
    df_count_vectorized = pd.DataFrame(count_matrix.toarray(), columns=count_feature_names)
    df_count_vectorized.index = dataframe.index

    # Instantiate TfidfVectorizer with specified parameters
    tfidf_vectorizer = TfidfVectorizer(
        stop_words='english',
        lowercase=True,
        ngram_range=(1, 1)
    )

    # Apply TfidfVectorizer
    tfidf_matrix = tfidf_vectorizer.fit_transform(dataframe['Free_Response_Text'])
    tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
    df_tfidf_vectorized = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_feature_names)
    df_tfidf_vectorized.index = dataframe.index

    return df_count_vectorized, df_tfidf_vectorized

In [ ]:
df_count_vectorized, df_tfidf_vectorized = vectorize_text_data(ML_Survey_Data)
df_tfidf_vectorized

In [ ]:
df_count_vectorized22, df_tfidf_vectorized22 = vectorize_text_data(ML_Survey_Data22)
df_count_vectorized22

## Combining Text Vectors with Structured Student Data

In [ ]:
ML_Survey_Data.iloc[:, :11]

In [ ]:
# Merge ML_Survey_Data and df_tfidf on their index values (SID)
df_merged_surv_features = pd.merge(ML_Survey_Data.iloc[:, :11], df_tfidf_vectorized, left_index=True, right_index=True)

# Display the first few rows of the merged DataFrame
print("First few rows of the merged DataFrame (Survey responses + tf-idf text features):")
display(df_merged_surv_features.head())

print(f"\nShape of the merged DataFrame: {df_merged_surv_features.shape}")

In [ ]:
# Merge ML_Survey_Data22 and df_tfidf on their index values (SID)
df_merged_surv_features22 = pd.merge(ML_Survey_Data22.iloc[:, :11], df_tfidf_vectorized22, left_index=True, right_index=True)

# Display the first few rows of the merged DataFrame
print("First few rows of the merged DataFrame (Survey responses + tf-idf text features):")
display(df_merged_surv_features22.head())

print(f"\nShape of the merged DataFrame: {df_merged_surv_features22.shape}")

In [ ]:
import os

# Define the base path for your Google Drive
drive_path = '/content/drive/MyDrive/'

# Define the target directory for exporting the merged features
export_dir = os.path.join(drive_path, 'IR ML Cert/MLCert Course 3/Course 3 Data')

# Create the directory if it doesn't exist
os.makedirs(export_dir, exist_ok=True)

# Define the full file path for the CSV
output_filename = 'ML_Survey_Data_Num.csv'
output_filepath = os.path.join(export_dir, output_filename)

# Export the DataFrame to CSV without the index (as SID is a column)
df_merged_surv_features.to_csv(output_filepath, index=False)

print(f"DataFrame successfully exported to: {output_filepath}")

In [ ]:
# Define the full file path for the CSV
output_filename = 'ML_Survey_Data22_Num.csv'
output_filepath = os.path.join(export_dir, output_filename)

# Export the DataFrame to CSV without the index (as SID is a column)
df_merged_surv_features22.to_csv(output_filepath, index=False)

print(f"DataFrame successfully exported to: {output_filepath}")